# Source-data coherence patch

Rewrites the incoherent cells in `gen_data/targets.csv`, `gen_data/climate_scenarios.csv`, and `gen_data/governance.csv` for **all five banks**, deriving every replacement from each bank's own data. It **backs up** the originals to a timestamped folder first, writes a `coherence_patch_changelog.json`, and is **safe to re-run** (a second run reports 0 changes).

**Decisions applied** (as agreed):
- Intensity-target baselines & milestones rescaled to each bank's computed financed-emissions intensity.
- Net-zero targets treated as **all-scopes**: baseline = operational baseline + financed emissions (2020 financed not in data, so the earliest available year is used as a proxy — noted in the changelog).
- Scenario `stranded_assets_estimate_meur` and `revenue_at_risk_meur` re-derived as `high_risk_exposure_pct x loans` and `x revenue` respectively, so neither can exceed its base.
- Board headcount percentages snapped to the nearest whole-director / whole-meeting value.

**Before running:** make sure `data_prep` has been run (so `financial_summary_clean.csv` exists with computed intensities) and that the paths in the CONFIG cell point at your `gen_data`. Then re-run `data_prep` afterwards so the payload picks up the corrected source, and the coherence gate I added should now report 0 errors.

In [1]:
# === SOURCE-DATA COHERENCE PATCH ============================================
# Rewrites the incoherent cells in gen_data/{targets,climate_scenarios,governance}.csv
# for ALL banks, deriving every value from each bank's own data. Originals are backed
# up first; a JSON change-log is written; safe to re-run (idempotent).
#
# Decisions applied (per the discussion):
#   - net-zero targets are treated as ALL-SCOPES  -> baseline = operational baseline + financed emissions
#   - board headcount percentages are snapped to the nearest whole-director value
import json, shutil, math
from pathlib import Path
from datetime import datetime
import pandas as pd

# ---- CONFIG ----------------------------------------------------------------
GEN_DATA        = "gen_data"                      # folder with the source CSVs
FIN_CLEAN_NAME  = "financial_summary_clean.csv"   # data_prep output (has computed intensity)
PROXY_NOTE      = "2020 financed emissions not in data; used earliest available year as proxy"

def _find(name, extra=()):
    for base in (Path("."), Path(GEN_DATA), Path(".."), Path("..")/GEN_DATA, *map(Path, extra)):
        f = base / name
        if f.exists(): return f
    raise FileNotFoundError(f"Could not locate {name}. Set its path in CONFIG.")

def _to_num(v):
    try:
        f = float(v); return f if f == f else None
    except Exception: return None

gd = Path(GEN_DATA)
targets    = pd.read_csv(_find("targets.csv"))
scenarios  = pd.read_csv(_find("climate_scenarios.csv"))
governance = pd.read_csv(_find("governance.csv"))
fin        = pd.read_csv(_find(FIN_CLEAN_NAME))

if fin["carbon_intensity_tco2e_per_meur_lending"].isna().all():
    raise RuntimeError(f"{FIN_CLEAN_NAME} has no computed intensity — run data_prep first.")

# per-bank lookups from the computed summary
INT, LOANS, REV = {}, {}, {}
for _, r in fin.iterrows():
    b, y = r["bank_id"], int(r["reporting_year"])
    INT[(b, y)]   = _to_num(r["carbon_intensity_tco2e_per_meur_lending"])
    LOANS[(b, y)] = _to_num(r["total_loans_meur"])
    REV[(b, y)]   = _to_num(r["total_revenue_meur"])
def _earliest(d, bank):
    ys = sorted(y for (b, y) in d if b == bank and d[(b, y)] is not None)
    return d[(bank, ys[0])] if ys else None
def intensity_for(bank, year):
    return INT.get((bank, int(year))) if (bank, int(year)) in INT else _earliest(INT, bank)
def loans_for(bank, year):
    return LOANS.get((bank, int(year))) if (bank, int(year)) in LOANS else _earliest(LOANS, bank)
def revenue_for(bank, year):
    return REV.get((bank, int(year))) if (bank, int(year)) in REV else _earliest(REV, bank)

changes = []
def log(file, bank, ref, field, old, new, reason):
    changes.append({"file": file, "bank": bank, "ref": ref, "field": field,
                    "old": old, "new": new, "reason": reason})

# ---- FIX 1+2: targets ------------------------------------------------------
def rescale_milestones(js, k):
    if not isinstance(js, str) or not js.strip(): return js
    try: ms = json.loads(js)
    except Exception: return js
    for m in ms:
        if isinstance(m, dict) and _to_num(m.get("value")) is not None:
            m["value"] = round(_to_num(m["value"]) * k, 2)
    return json.dumps(ms)

for i, r in targets.iterrows():
    bank, tid = r["bank_id"], r["target_id"]
    ttype = str(r.get("target_type", "")).lower()
    metric = str(r.get("metric", "")).lower()
    by = int(r["baseline_year"]) if _to_num(r["baseline_year"]) else None
    old_bv = _to_num(r["baseline_value"])

    # 1) intensity-target baseline -> bank's computed intensity in the baseline year
    if "intensity" in ttype or "per_meur" in metric:
        comp = intensity_for(bank, by)
        if comp and old_bv:
            new_bv = round(comp, 2)
            k = new_bv / old_bv
            if old_bv != new_bv:
                targets.at[i, "baseline_value"] = new_bv
                log("targets.csv", bank, tid, "baseline_value", old_bv, new_bv,
                    f"intensity target must match computed financed-emissions intensity ({by})")
            old_ms = r.get("interim_milestones_json")
            new_ms = rescale_milestones(old_ms, k)
            if isinstance(old_ms, str) and new_ms != old_ms:
                targets.at[i, "interim_milestones_json"] = new_ms
                log("targets.csv", bank, tid, "interim_milestones_json", old_ms, new_ms,
                    f"milestones rescaled by x{k:.3f} to preserve the reduction path")

    # 2) net-zero all-scopes baseline -> operational baseline + financed emissions
    if "net_zero" in ttype or str(r.get("scope", "")).lower() == "all_scopes":
        op = targets[(targets["bank_id"] == bank) &
                     (targets["target_type"].astype(str).str.lower() == "absolute_reduction")]
        op_bv = _to_num(op["baseline_value"].iloc[0]) if not op.empty else 0.0
        fin_proxy = None
        inten, ln = intensity_for(bank, by), loans_for(bank, by)
        if inten is not None and ln is not None:
            fin_proxy = inten * ln
        if fin_proxy is not None:
            new_bv = round((op_bv or 0.0) + fin_proxy, 0)
            if old_bv != new_bv:
                targets.at[i, "baseline_value"] = new_bv
                log("targets.csv", bank, tid, "baseline_value", old_bv, new_bv,
                    f"all-scopes net-zero baseline = operational ({op_bv:.0f}) + financed (~{fin_proxy:.0f}); {PROXY_NOTE}")

# ---- FIX 3: climate_scenarios ---------------------------------------------
for i, r in scenarios.iterrows():
    bank = r["bank_id"]
    hr = _to_num(r.get("high_risk_exposure_pct"))
    yr = r.get("analysis_conducted_year") or r.get("horizon_year") or 2024
    ln, rev = loans_for(bank, yr), revenue_for(bank, yr)
    if hr is None: continue
    if "stranded_assets_estimate_meur" in scenarios.columns and ln is not None:
        old = _to_num(r.get("stranded_assets_estimate_meur"))
        new = round(hr / 100.0 * ln, 1)
        if old != new:
            scenarios.at[i, "stranded_assets_estimate_meur"] = new
            log("climate_scenarios.csv", bank, r.get("scenario_id"), "stranded_assets_estimate_meur",
                old, new, "bounded: high_risk_exposure_pct x total_loans (cannot strand more than held)")
    if "revenue_at_risk_meur" in scenarios.columns and rev is not None:
        old = _to_num(r.get("revenue_at_risk_meur"))
        new = round(hr / 100.0 * rev, 1)
        if old != new:
            scenarios.at[i, "revenue_at_risk_meur"] = new
            log("climate_scenarios.csv", bank, r.get("scenario_id"), "revenue_at_risk_meur",
                old, new, "bounded: high_risk_exposure_pct x total_revenue (cannot exceed revenue)")

# ---- FIX 4: governance -----------------------------------------------------
def snap(pct, denom):
    pct, denom = _to_num(pct), _to_num(denom)
    if pct is None or not denom: return pct
    step = 100.0 / denom
    return round(round(pct / step) * step, 1)

for i, r in governance.iterrows():
    bank, yr = r["bank_id"], r.get("reporting_year")
    bs = r.get("board_size")
    for col, denom, why in [("independent_directors_pct", bs, "whole directors / board_size"),
                            ("board_climate_expertise_pct", bs, "whole directors / board_size"),
                            ("climate_on_board_agenda_pct", r.get("board_full_meeting_frequency"),
                             "whole meetings / meeting frequency")]:
        if col in governance.columns:
            old = _to_num(r.get(col)); new = snap(old, denom)
            if old is not None and new is not None and old != new:
                governance.at[i, col] = new
                log("governance.csv", bank, yr, col, old, new, f"snapped to {why}")

# ---- backup, write, log ----------------------------------------------------
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup = gd / f"_backup_pre_coherence_fix_{stamp}"
backup.mkdir(parents=True, exist_ok=True)
for name, df in [("targets.csv", targets), ("climate_scenarios.csv", scenarios), ("governance.csv", governance)]:
    src = _find(name)
    shutil.copy2(src, backup / name)
    df.to_csv(src, index=False)

Path("coherence_patch_changelog.json").write_text(json.dumps(changes, indent=2, default=str))

print(f"Applied {len(changes)} cell change(s). Originals backed up to: {backup}")
from collections import Counter
for (f, fld), n in Counter((c["file"], c["field"]) for c in changes).items():
    print(f"  {f:24s} {fld:32s} {n:>3d} cell(s)")
print("\nExamples (BANK01):")
for c in [c for c in changes if c["bank"] == "BANK01"][:6]:
    o = c["old"]; nw = c["new"]
    o = (o[:40] + "...") if isinstance(o, str) and len(o) > 43 else o
    nw = (nw[:40] + "...") if isinstance(nw, str) and len(nw) > 43 else nw
    print(f"  {c['ref']} {c['field']}: {o}  ->  {nw}")
print("\nFull detail in coherence_patch_changelog.json")


Applied 228 cell change(s). Originals backed up to: gen_data\_backup_pre_coherence_fix_20260618_145949
  targets.csv              baseline_value                    10 cell(s)
  targets.csv              interim_milestones_json            5 cell(s)
  climate_scenarios.csv    stranded_assets_estimate_meur     84 cell(s)
  climate_scenarios.csv    revenue_at_risk_meur              84 cell(s)
  governance.csv           independent_directors_pct         15 cell(s)
  governance.csv           board_climate_expertise_pct       15 cell(s)
  governance.csv           climate_on_board_agenda_pct       15 cell(s)

Examples (BANK01):
  TGT002 baseline_value: 20.0  ->  1173.34
  TGT002 interim_milestones_json: [{"year": 2025, "value": 18.2, "metric":...  ->  [{"year": 2025, "value": 1067.74, "metri...
  TGT003 baseline_value: 120000.0  ->  36688026.0
  SCN0001 stranded_assets_estimate_meur: 9762.6  ->  10871.6
  SCN0001 revenue_at_risk_meur: 9689.2  ->  8423.5
  SCN0002 stranded_assets_estimate_meur: 